In [2]:
!pip install pandas

  Using cached pandas-3.0.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached numpy-2.4.6-cp314-cp314-macosx_14_0_arm64.whl.metadata (6.6 kB)
Using cached pandas-3.0.3-cp314-cp314-macosx_11_0_arm64.whl (9.9 MB)
Using cached numpy-2.4.6-cp314-cp314-macosx_14_0_arm64.whl (5.2 MB)

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import pandas as pd

df = pd.read_csv('global_ads_performance_dataset.csv')

display(df.head())

print("\n--- Data Information ---")
df.info()


,date,platform,campaign_type,industry,country,impressions,clicks,CTR,CPC,ad_spend,conversions,CPA,revenue,ROAS
0,2024-01-21,Google Ads,Search,Fintech,UAE,59886,2113,0.0353,1.26,2662.38,159,16.74,4803.43,1.80
1,2024-01-22,TikTok Ads,Search,EdTech,UK,135608,5220,0.0385,1.18,6159.60,411,14.99,64126.68,10.41
2,2024-06-15,TikTok Ads,Video,Healthcare,USA,92313,5991,0.0649,0.85,5092.35,267,19.07,10489.07,2.06
3,2024-01-02,TikTok Ads,Shopping,SaaS,Germany,83953,5935,0.0707,1.32,7834.20,296,26.47,50505.07,6.45
4,2024-02-22,TikTok Ads,Search,Healthcare,UK,91807,4489,0.0489,1.93,8663.77,107,80.97,3369.53,0.39



--- Data Information ---
<class 'pandas.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           1800 non-null   str    
 1   platform       1800 non-null   str    
 2   campaign_type  1800 non-null   str    
 3   industry       1800 non-null   str    
 4   country        1800 non-null   str    
 5   impressions    1800 non-null   int64  
 6   clicks         1800 non-null   int64  
 7   CTR            1800 non-null   float64
 8   CPC            1800 non-null   float64
 9   ad_spend       1800 non-null   float64
 10  conversions    1800 non-null   int64  
 11  CPA            1800 non-null   float64
 12  revenue        1800 non-null   float64
 13  ROAS           1800 non-null   float64
dtypes: float64(6), int64(3), str(5)
memory usage: 197.0 KB


In [4]:

if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'])

df = df.fillna(0)

df = df.drop_duplicates()

print("Dates standardized, nulls handled, and duplicates removed!")

Dates standardized, nulls handled, and duplicates removed!


In [5]:

df.to_csv('cleaned_marketing_roi.csv', index=False)

print("Success! 'cleaned_marketing_roi.csv' is ready for SQL.")

Success! 'cleaned_marketing_roi.csv' is ready for SQL.


In [6]:
import sqlite3
import pandas as pd

clean_df = pd.read_csv('cleaned_marketing_roi.csv')

conn = sqlite3.connect('marketing_data.db')

clean_df.to_sql('campaign_data', conn, if_exists='replace', index=False)

print("SQL Database connected and 'campaign_data' table created!")

SQL Database connected and 'campaign_data' table created!


In [7]:
query = """
SELECT
    platform,
    SUM(ad_spend) AS Total_Spend,
    SUM(revenue) AS Total_Revenue,
    (SUM(revenue) - SUM(ad_spend)) AS Net_Profit,
    ROUND(((SUM(revenue) - SUM(ad_spend)) / SUM(ad_spend)) * 100, 2) AS ROI_Percentage,
    ROUND(SUM(ad_spend) / SUM(conversions), 2) AS Cost_Per_Acquisition
FROM
    campaign_data
GROUP BY
    platform
ORDER BY
    ROI_Percentage DESC;
"""

sql_results = pd.read_sql(query, conn)
display(sql_results)

,platform,Total_Spend,Total_Revenue,Net_Profit,ROI_Percentage,Cost_Per_Acquisition
0,TikTok Ads,2653418.51,20223540.07,17570121.56,662.17,21.67
1,Meta Ads,2106061.67,11926045.79,9819984.12,466.27,28.75
2,Google Ads,6349268.91,22033744.95,15684476.04,247.03,48.43
